# 08 Inference Optimization: PyTorch KV Cache Memory & Roofline Profiler

Calculating PyTorch KV Cache VRAM sizes and plotting the GPU Roofline Model.

## Part 1: PyTorch KV Cache Memory Footprint Calculation
Using PyTorch `tensor.element_size()` to measure VRAM consumption across MHA, MQA, and GQA.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('dark_background')

def get_kv_cache_bytes(layers, h_kv, d_head, seq_len, batch_size, dtype=torch.float16):
    element_size = torch.tensor([], dtype=dtype).element_size()
    total_bytes = 2 * element_size * layers * h_kv * d_head * seq_len * batch_size
    return total_bytes

seq_lengths = np.array([1024, 4096, 16384, 32768, 65536, 131072])
L, d_head, P, B = 80, 128, 2, 1

vram_mha = [get_kv_cache_bytes(L, 64, d_head, T, B) / (1024**3) for T in seq_lengths]
vram_gqa = [get_kv_cache_bytes(L, 8, d_head, T, B) / (1024**3) for T in seq_lengths]
vram_mqa = [get_kv_cache_bytes(L, 1, d_head, T, B) / (1024**3) for T in seq_lengths]

plt.figure(figsize=(9, 4.5))
plt.plot(seq_lengths / 1024, vram_mha, marker='o', label='MHA (64 KV Heads)', color='#ff6b6b')
plt.plot(seq_lengths / 1024, vram_gqa, marker='s', label='GQA (8 KV Heads - Llama-3)', color='#51cf66')
plt.plot(seq_lengths / 1024, vram_mqa, marker='^', label='MQA (1 KV Head)', color='#4dabf7')
plt.axhline(80, color='#ff922b', linestyle='--', label='80 GB VRAM Threshold (Single A100 GPU)')
plt.xscale('log', base=2); plt.yscale('log')
plt.title('70B Model PyTorch KV Cache VRAM Footprint vs Context Length')
plt.xlabel('Context Length (k Tokens)'); plt.ylabel('KV Cache VRAM (GB)')
plt.grid(True, which='both'); plt.legend(); plt.show()

## Part 2: Roofline Performance Model (Prefill vs Decode & FlashAttention IO Shift)
Plotting operational intensity (FLOP/byte) vs attainable performance (TFLOP/s) on A100 GPU hardware bounds.

In [ ]:
intensity = np.logspace(-1, 3, 200)
peak_bandwidth, peak_compute = 2000.0, 312.0
mem_bound = intensity * (peak_bandwidth / 1000.0)
roofline = np.minimum(mem_bound, peak_compute)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(intensity, roofline, color='#ffffff', lw=2.5, label='GPU Hardware Roofline Bound (A100 FP16)')
ax.scatter([120], [300], color='#51cf66', s=150, zorder=5, label='Prefill Phase (Compute-Bound)')
ax.scatter([0.8], [1.6], color='#ff6b6b', s=150, zorder=5, label='Decode Phase (Memory-Bound)')
ax.scatter([8.0], [16.0], color='#4dabf7', s=150, zorder=5, label='Decode with FlashAttention (IO Tiling)')
ax.annotate('', xy=(7.0, 14.0), xytext=(1.2, 2.4), arrowprops=dict(facecolor='#4dabf7', shrink=0.05, width=2, headwidth=7))
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_title('Roofline Performance Model: LLM Prefill vs Decode & FlashAttention IO Shift')
ax.set_xlabel('Operational Intensity (FLOPs / Byte Transferred from HBM)'); ax.set_ylabel('Attainable Performance (TFLOP/s)')
ax.grid(True, which='both'); ax.legend(fontsize=8.5, loc='upper left'); plt.show()